# Phase 3: Nystroem RBF SVM (full training set)

Exact `SVC(kernel="rbf")` does not scale to ~90k flows. This notebook keeps a **Gaussian RBF kernel**, but approximates it with **`sklearn.kernel_approximation.Nystroem`**, then fits a linear model in that feature space on the **full** training split.

Pipeline:
`RobustScaler impute/scale → Nystroem(RBF) → LinearSVC → CalibratedClassifierCV (probabilities)`

We tune `C`, RBF `gamma`, and Nystroem `n_components`, then pick a validation F1 threshold.


## Setup and load splits

In [ ]:
import json
import sys
import time
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve, make_scorer
)

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from src.data.flow_preprocessing import ATTACK_COL, BINARY_COL, FLOW_KEY_COL, TARGET_COL, build_flow_preprocessor

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "phase3"
metadata = json.loads((DATA_DIR / "flow_split_metadata.json").read_text())
train_df = pd.read_csv(DATA_DIR / "flow_train.csv", low_memory=False)
validation_df = pd.read_csv(DATA_DIR / "flow_validation.csv", low_memory=False)
test_df = pd.read_csv(DATA_DIR / "flow_test.csv", low_memory=False)
print(train_df.shape, validation_df.shape, test_df.shape)

In [ ]:
feature_cols = metadata["feature_columns"]
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].to_numpy()
X_validation = validation_df[feature_cols]
y_validation = validation_df[TARGET_COL].to_numpy()
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].to_numpy()
print("features:", len(feature_cols), "train attack rate:", float(y_train.mean()))

## Tune Nystroem RBF + LinearSVC

`dual=False` is appropriate when transformed samples ≫ features. Calibration gives `predict_proba` for threshold/PR curves.


In [ ]:
RANDOM_STATE = 1
N_ITER = 16

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

base_svm = LinearSVC(
    class_weight="balanced",
    dual=False,
    max_iter=20000,
    random_state=RANDOM_STATE,
)
calibrated = CalibratedClassifierCV(estimator=base_svm, method="sigmoid", cv=2)

svm_pipeline = Pipeline([
    ("preprocessor", build_flow_preprocessor(scale=True)),
    ("nystroem", Nystroem(kernel="rbf", random_state=RANDOM_STATE)),
    ("model", calibrated),
])

param_distributions = {
    "nystroem__n_components": [100, 200, 300, 400, 500],
    "nystroem__gamma": [0.0005, 0.001, 0.01, 0.05, 0.1, 0.2, None],
    "model__estimator__C": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
}

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
}

svm_search = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_STATE,
    return_train_score=False,
)

start_search = time.perf_counter()
svm_search.fit(X_train, y_train)
search_time = time.perf_counter() - start_search
svm_pipeline = svm_search.best_estimator_

print(f"Search time: {search_time:.1f}s")
print(f"Best CV PR-AUC: {svm_search.best_score_:.4f}")
for k, v in svm_search.best_params_.items():
    print(f"  {k}: {v}")

In [ ]:
cv_results_df = pd.DataFrame(svm_search.cv_results_)
cols = [
    "rank_test_pr_auc", "mean_test_pr_auc", "mean_test_roc_auc", "mean_test_f1",
    "mean_test_precision", "mean_test_recall", "mean_fit_time",
    "param_nystroem__n_components", "param_nystroem__gamma", "param_model__estimator__C",
]
display(cv_results_df[cols].sort_values("rank_test_pr_auc").head(10).round(4))

## Threshold tuning and evaluation

In [ ]:
def evaluate_binary_classifier(name, y_true, scores, threshold, prediction_time):
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "dataset": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "fnr": fn / (fn + tp) if (fn + tp) else 0.0,
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "alerts": int(y_pred.sum()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "prediction_time_sec": prediction_time,
    }, y_pred


def best_f1_threshold(y_true, scores):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": 0.5, "f1": -1.0}
    rows = []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        rows.append({
            "threshold": float(t),
            "f1": f1,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
        })
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1)}
    return best, pd.DataFrame(rows)

start = time.perf_counter()
validation_scores = svm_pipeline.predict_proba(X_validation)[:, 1]
validation_prediction_time = time.perf_counter() - start
threshold_info, threshold_curve = best_f1_threshold(y_validation, validation_scores)
CLASSIFICATION_THRESHOLD = threshold_info["threshold"]
print(f"Chosen threshold: {CLASSIFICATION_THRESHOLD:.4f} (val F1={threshold_info['f1']:.4f})")

start = time.perf_counter()
test_scores = svm_pipeline.predict_proba(X_test)[:, 1]
test_prediction_time = time.perf_counter() - start

validation_metrics, validation_pred = evaluate_binary_classifier(
    "Validation", y_validation, validation_scores, CLASSIFICATION_THRESHOLD, validation_prediction_time
)
test_metrics, test_pred = evaluate_binary_classifier(
    "Test", y_test, test_scores, CLASSIFICATION_THRESHOLD, test_prediction_time
)
metrics_df = pd.DataFrame([validation_metrics, test_metrics])
display(metrics_df.round(4))

plt.figure(figsize=(7, 4))
plt.plot(threshold_curve["threshold"], threshold_curve["f1"], label="F1")
plt.plot(threshold_curve["threshold"], threshold_curve["precision"], label="Precision", alpha=0.8)
plt.plot(threshold_curve["threshold"], threshold_curve["recall"], label="Recall", alpha=0.8)
plt.axvline(CLASSIFICATION_THRESHOLD, linestyle="--", color="black")
plt.xlabel("Threshold"); plt.ylabel("Score"); plt.title("Nystroem RBF SVM val threshold sweep")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    plt.imshow(cm); plt.title(title); plt.colorbar()
    plt.xticks([0, 1], ["Benign", "Attack"]); plt.yticks([0, 1], ["Benign", "Attack"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

plot_confusion_matrix(y_validation, validation_pred, "Nystroem RBF SVM Validation CM")
plot_confusion_matrix(y_test, test_pred, "Nystroem RBF SVM Test CM")

display(pd.DataFrame(classification_report(
    y_test, test_pred, target_names=["Benign", "Attack"], output_dict=True, zero_division=0
)).T.round(4))

type_results = test_df[[ATTACK_COL]].copy()
type_results["y_true"] = y_test
type_results["y_pred"] = test_pred
type_results["correct"] = type_results["y_true"] == type_results["y_pred"]
result_by_type = type_results.groupby(ATTACK_COL).agg(total=("correct", "size"), correct=("correct", "sum"))
result_by_type["correct_rate"] = 100 * result_by_type["correct"] / result_by_type["total"]
display(result_by_type.reindex(
    ["Benign", "DDoS-HTTP Flood", "DoS-HTTP Flood", "DNS Spoofing", "Brute Force", "XSS"]
).dropna().round(2))

fpr, tpr, _ = roc_curve(y_test, test_scores)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC={roc_auc_score(y_test, test_scores):.4f}")
plt.plot([0, 1], [0, 1], "--"); plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("Nystroem RBF SVM Test ROC"); plt.legend(); plt.grid(True); plt.show()

p, r, _ = precision_recall_curve(y_test, test_scores)
plt.figure(figsize=(6, 5))
plt.plot(r, p, label=f"PR-AUC={average_precision_score(y_test, test_scores):.4f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Nystroem RBF SVM Test PR"); plt.legend(); plt.grid(True); plt.show()

## Save results

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "phase3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

legacy = RESULTS_DIR / "svm_tuned_metrics.csv"
if legacy.exists():
    old = pd.read_csv(legacy)
    model_name = str(old.iloc[0]["model"]) if "model" in old.columns else ""
    if "RBF SVM" in model_name and "Nystroem" not in model_name:
        old.to_csv(RESULTS_DIR / "svm_rbf_subsample_metrics.csv", index=False)

metrics_output = metrics_df.copy()
metrics_output.insert(0, "model", "Nystroem RBF SVM")
metrics_output["search_time_sec"] = search_time
metrics_output["refit_time_sec"] = svm_search.refit_time_
metrics_output["train_rows"] = len(X_train)
metrics_output.to_csv(RESULTS_DIR / "svm_tuned_metrics.csv", index=False)
cv_results_df.to_csv(RESULTS_DIR / "svm_random_search_results.csv", index=False)

best_parameters = {
    "method": "nystroem_rbf_linearsvc",
    "kernel": "rbf",
    "train_rows": int(len(X_train)),
    "best_cv_pr_auc": float(svm_search.best_score_),
    "search_time_sec": search_time,
    "refit_time_sec": svm_search.refit_time_,
    "n_iter": N_ITER,
    "cv_folds": cv.get_n_splits(),
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "validation_threshold_f1": threshold_info["f1"],
    "best_params": {k: (float(v) if hasattr(v, "item") else v) for k, v in svm_search.best_params_.items()},
}
(RESULTS_DIR / "svm_best_parameters.json").write_text(json.dumps(best_parameters, indent=2))

pred = test_df[[FLOW_KEY_COL, ATTACK_COL, BINARY_COL, TARGET_COL]].copy()
pred["attack_probability"] = test_scores
pred["prediction"] = test_pred
pred.to_csv(RESULTS_DIR / "svm_tuned_test_predictions.csv", index=False)

test_row = metrics_df.loc[metrics_df["dataset"] == "Test"].iloc[0]
print("Final Nystroem RBF SVM test results")
print("-----------------------------------")
print(f"Best CV PR-AUC: {svm_search.best_score_:.4f}")
print(f"Threshold: {CLASSIFICATION_THRESHOLD:.4f}")
for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "fpr", "fnr"]:
    print(f"{k}: {test_row[k]:.4f}")
print("Best params:", svm_search.best_params_)
print("Saved to", RESULTS_DIR)